# Pipeline de Détection de Fausses Nouvelles (Branche Knowledge)

## Vue d'ensemble du notebook

**Objectif:** Orchestre un pipeline complet de vérification factuelle basée sur la connaissance:
1. **Setup de l'environnement** (dépendances, datasets, modèles)
2. **Entraînement du Claim Detector** (détection de claims avec DistilBERT)
3. **Initialisation du pipeline** (Evidence Retriever + Claim Verifier)
4. **Évaluation complète** (test sur dataset FEVER)
5. **Pipeline de démonstration** (vérification de texte complet)

**Environnement:** Linux/Mac/Windows avec GPU optionnelle

---

## Architecture du Pipeline

```
Texte d'entrée
   ↓
Phase 1: Setup Environnement
├─ Installation des dépendances
├─ Téléchargement des datasets
└─ Configuration des APIs
   ↓
Phase 2: Entraînement Claim Detector
├─ Chargement de groundtruth.csv
├─ Fine-tuning DistilBERT (3 epochs)
├─ Evaluation (Precision, Recall, F1)
└─ Sauvegarde du modèle
   ↓
Phase 3: Initialisation du Pipeline
├─ Evidence Retriever (Google Search, Wolfram Alpha, spaCy NER)
├─ Claim Verifier (RoBERTa NLI)
└─ Tests simples
   ↓
Phase 4: Évaluation Complète
├─ Chargement dataset FEVER
├─ Équilibrage
├─ Test le retriever
├─ Test du verifier
└─ Matrice de confusion + Rapport
   ↓
Phase 5: Pipeline Complet
├─ Traitement texte
├─ Extraction de claims & entités
├─ Récupération de preuves
├─ Vérification (supported/refuted/neutral)
└─ Rapport JSON
```

---

## Phases d'exécution

### Phase 1: Setup Environnement
**Durée:** 5-10 minutes
- Installation des packages Python
- Téléchargement des modèles spaCy
- Téléchargement du dataset groundtruth.csv
- Vérification des configurations

### Phase 2: Entraînement Claim Detector
**Durée:** 5-15 minutes (GPU), 20-40 min (CPU)
- Dataset: groundtruth.csv (~5000 samples)
- Modèle: distilbert-base-uncased
- Learning rate: 2e-5, Epochs: 3, Batch: 16
- Sortie: modèle sauvegardé + métriques

### Phase 3: Initialisation Pipeline
**Durée:** 2-3 minutes
- Configuration du Evidence Retriever
- Configuration du Claim Verifier
- Tests simples multi-langues
- Vérification des APIs

### Phase 4: Évaluation Complète
**Durée:** 10-20 minutes
- Dataset: 90 samples FEVER équilibrés (30 par classe)
- Classe 1: SUPPORTED (claims supportés par les preuves)
- Classe 2: REFUTED (claims contredits par les preuves)
- Classe 3: NEUTRAL (pas assez d'informations)
- Outputs: matrice de confusion, rapport détaillé

### Phase 5: Pipeline Complet
**Durée:** Variable (selon APIs)
- Entrée: texte/article complet
- Sortie: rapport JSON avec verdicts
- Languages: EN, FR, ES supportées

---

## Fichiers générés

| Fichier | Créé par | Utilisation |
|---------|----------|------------|
| `groundtruth.csv` | setup_environment.py | Dataset pour claim detector |
| `my_claim_model/` | train_claim_detector.py | Modèle fine-tuné |
| `results/confusion_matrix.png` | evaluate_pipeline.py | Visualisation |
| `results/evaluation_report.txt` | evaluate_pipeline.py | Rapport d'évaluation |
| `results/verification_report_*.json` | full_pipeline.py | Rapport de vérification |

---

## Exécution du pipeline

Exécutez chaque cellule **dans l'ordre**.


## Phase 1: Setup de l'environnement

Configuration initiale et téléchargement des ressources nécessaires.

In [7]:
!python setup_environment.py

🚀 SETUP ENVIRONMENT - KNOWLEDGE BRANCH

🎮 Device : CUDA - NVIDIA GeForce RTX 4060
💾 GPU VRAM disponible : 8.2 GB

📂 Configuration des chemins...
   Répertoire de travail : /home/juul/Documents/IFT714 Traitement des LN/Projet/Detection_fake_news/knowledge_branch
   ✅ Projet trouvé : True
   ✅ Data trouvée : True

📦 Vérification des dépendances...
   ✅ transformers
   ✅ torch
   ✅ spacy
   ✅ pandas
   ✅ datasets
   ✅ scikit-learn
   ❌ wikipedia-api - À installer

📥 Installation de 1 package(s)...
   ✅ Installation terminée

🔤 Téléchargement des modèles spaCy...
   ✅ en_core_web_sm
   ✅ fr_core_news_sm
   ✅ es_core_news_sm

📥 Téléchargement du dataset...
   ✅ Fichier déjà présent : /home/juul/Documents/IFT714 Traitement des LN/Projet/Detection_fake_news/knowledge_branch/groundtruth.csv
🔍 Vérification des modules...
   ✅ evidence_retrieval
   ✅ claim_verification
   ✅ claim_detection

✅ SETUP TERMINÉ


## Phase 2: Entraînement du Claim Detector

Fine-tuning d'un DistilBERT pour détecter si un texte est un "claim" (énoncé factuellement vérifiable).
**Note:** Cette phase prend 5-15 minutes selon votre GPU. Vous pouvez la sauter si le modèle existe déjà.

In [8]:
!python train_claim_detector_partA.py

🚀 TRAINING CLAIM DETECTOR (80% Part A)

📂 Chargement du dataset groundtruth_partA.csv (80% Part A)...
   📊 Dimensions : (780, 10)
   📋 Colonnes : ['Sentence_id', 'Text', 'Speaker', 'Speaker_title', 'Speaker_party', 'File_id', 'Length', 'Line_number', 'Sentiment', 'Verdict']
   📈 Distribution des labels :
Verdict
-1    590
 1    190
Name: count, dtype: int64
   ✅ Dataset préparé

✂️  Division du dataset...
   Train : 624 samples
   Test  : 156 samples

🤖 Fine-tuning du Claim Detector...
   Device : CPU (pour éviter les problèmes de mémoire GPU)

   tokenization en cours...
Map: 100%|██████████████████████████| 156/156 [00:00<00:00, 23581.34 examples/s]
   Chargement du modèle...
Loading weights: 100%|█████████████████████| 100/100 [00:00<00:00, 18906.89it/s]
DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECT

## Phase 3: Initialisation du Pipeline

Configuration et test des composants principaux:
- Evidence Retriever (récupération de preuves)
- Claim Verifier (vérification NLI)
- Claim Detector optionnel (si disponible)
- Tests multi-langues (EN, FR, ES)

In [9]:
!python initialize_pipeline.py

🚀 INITIALIZE KNOWLEDGE BRANCH

🔍 Initialisation du Evidence Retriever...
   ✅ Module evidence_retrieval importé
   Configuration :
      - Wolfram Alpha : ✅
      - Google API : ❌ (optionnel)
      - Google CSE : ✅
   ✅ Evidence Retriever initialisé

🔐 Initialisation du Claim Verifier...
   ✅ Module claim_verification importé
Loading weights: 100%|█████████████████████| 202/202 [00:00<00:00, 26283.52it/s]
DebertaV2ForSequenceClassification LOAD REPORT from: MoritzLaurer/DeBERTa-v3-base-mnli-fever-anli
Key                             | Status     |  | 
--------------------------------+------------+--+-
deberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
   ✅ Claim Verifier initialisé

💾 Sauvegarde de la configuration...
   ✅ Configuration définie

🧪 Test du Evidence Retriever...
   Résultats des recherches :
   ✅ [Eiffel Tower] (en)
      Phrase : The Eiffel Tower is 3

## Phase 4: Évaluation Complète

Teste le pipeline complet sur le dataset FEVER équilibré (90 samples):
- 30 samples SUPPORTED (claims confirmés par les preuves)
- 30 samples REFUTED (claims contredits par les preuves)
- 30 samples NEUTRAL / NOT ENOUGH INFO (informations insuffisantes)

Génère:
- Rapport de classification (Precision, Recall, F1-score)
- Matrice de confusion
- Analyse de récupération par classe

In [10]:
!python evaluate_pipeline_partA.py

🚀 EVALUATION - KNOWLEDGE BRANCH PIPELINE (80% Part A)

✅ Modules importés

📂 Chargement du dataset FEVER train_partA.jsonl (80% Part A)...
   📊 Total : 81962 instances
   📈 Distribution des labels :
label
SUPPORTED                    59081
REFUTED                      22631
NEUTRAL / NOT ENOUGH INFO      250
Name: count, dtype: int64

⚖️  Équilibrage du dataset (30 par classe)...
   ✅ Dataset équilibré : 90 instances
   📈 Distribution :
label
REFUTED                      30
SUPPORTED                    30
NEUTRAL / NOT ENOUGH INFO    30
Name: count, dtype: int64

🔧 Initialisation des composants...
Loading weights: 100%|█████████████████████| 202/202 [00:00<00:00, 20381.76it/s]
DebertaV2ForSequenceClassification LOAD REPORT from: MoritzLaurer/DeBERTa-v3-base-mnli-fever-anli
Key                             | Status     |  | 
--------------------------------+------------+--+-
deberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from differen

## Phase 5: Pipeline Complet - Démonstration

Applique le pipeline complet à un texte arbitraire:
1. Extraction des phrases
2. Détection de claims avec spaCy NER et optionnellement ML
3. Récupération de preuves
4. Vérification avec NLI
5. Génération d'un rapport JSON

Langues supportées: EN, FR, ES

In [11]:
!python full_pipeline.py

🚀 PIPELINE COMPLET - FACT-CHECKING

🔧 Initialisation du pipeline complet...
Loading weights: 100%|█████████████████████| 202/202 [00:00<00:00, 18390.08it/s]
DebertaV2ForSequenceClassification LOAD REPORT from: MoritzLaurer/DeBERTa-v3-base-mnli-fever-anli
Key                             | Status     |  | 
--------------------------------+------------+--+-
deberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Loading weights: 100%|█████████████████████| 104/104 [00:00<00:00, 17813.12it/s]
   ✅ Pipeline initialisé

📝 Texte à analyser (en):

        Algeria is located in Africa. The Eiffel Tower is in Paris. 
        The Eiffel Tower was built in 1990. Paris is the capital of France.
        Mount Everest is the highest mountain in the world.
        


🚀 --- TRAITEMENT DU TEXTE (EN) ---

[1] Algeria is located in Africa.
    ℹ️  ML Score: 0.994
    🏷️  Entités: GPE: Algeri

## Bonus: Test Multilingue

Lance le pipeline sur des textes en anglais, français et espagnol.

In [12]:
from full_pipeline import test_multiple_languages

test_multiple_languages()

🌍 TEST MULTILINGUE

🔧 Initialisation du pipeline complet...


Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

DebertaV2ForSequenceClassification LOAD REPORT from: MoritzLaurer/DeBERTa-v3-base-mnli-fever-anli
Key                             | Status     |  | 
--------------------------------+------------+--+-
deberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

   ✅ Pipeline initialisé


--- EN ---

🚀 --- TRAITEMENT DU TEXTE (EN) ---

[1] The Great Wall of China is one of the largest structures.
    ℹ️  ML Score: 0.996
    🔍 Vérification en cours...
    ✅ Source: Great Wall of China
    📄 Preuve: The Great Wall of China (traditional Chinese: 萬里長城; simplified Chinese: 万里长城; pi...
    ✅ Verdict: NEUTRAL / NOT ENOUGH INFO (confiance: 0.998)

[2] Paris is in France.

    🏷️  Entités: GPE: Paris, France
    🔍 Vérification en cours...
    ✅ Source: WolframAlpha
    📄 Preuve: Paris is the capital and largest city of France, with an estimated city populati...
    ✅ Verdict: SUPPORTED (confiance: 0.999)


📊 RÉSUMÉ DU RAPPORT

Total d'items traités : 2

Verdicts :
  NEUTRAL / NOT ENOUGH INFO :   1 ( 50.0%)
  SUPPORTED            :   1 ( 50.0%)

Détails :
  1. [NEUTRAL / NOT ENOUGH INFO] The Great Wall of China is one of the largest structures.... (conf: 0.998)
  2. [SUPPORTED      ] Paris is in France.... (conf: 0.999)


--- FR ---

🚀 --- TRAITEMENT DU

{'en': [{'sentence': 'The Great Wall of China is one of the largest structures.',
   'verdict': 'NEUTRAL / NOT ENOUGH INFO',
   'confidence': 0.9981830716133118,
   'source': 'Great Wall of China',
   'source_url': 'https://en.wikipedia.org/wiki/Great_Wall_of_China',
   'entities': {},
   'ml_score': 0.9960747957229614,
   'evidence_snippet': 'The Great Wall of China (traditional Chinese: 萬里長城; simplified Chinese: 万里长城; pinyin: Wànlǐ Chángchéng, literally "ten thousand li long wall") is a se'},
  {'sentence': 'Paris is in France.',
   'verdict': 'SUPPORTED',
   'confidence': 0.9989759922027588,
   'source': 'WolframAlpha',
   'source_url': 'https://www.wolframalpha.com',
   'entities': {'GPE': ['Paris', 'France']},
   'ml_score': None,
   'evidence_snippet': 'Paris is the capital and largest city of France, with an estimated city population of 2,048,472 in an area of 105.4 km^2 (40.7 sq mi), and a metropoli'}],
 'fr': [{'sentence': "La France est un pays d'Europe.",
   'verdict': 'NOT_